# Global Markets, Local Signals: What Actually Drives a BUY Rating?

**A cross-asset intelligence notebook with high-signal EDA, a clean baseline, and a leakage-aware model to predict analyst conviction.**

---

## Introduction

Most finance datasets force you to choose between breadth and usability. This one does not.

In a single table, we get **451 tickers across equities, ETFs, crypto, forex, commodities, and indices**, together with a rich mix of:

- fundamentals
- technical indicators
- risk metrics
- proprietary composite scores

That makes the dataset useful for much more than a few standard plots. The real opportunity is to ask:

> **What separates strong market profiles from weak ones across asset classes, and how much of analyst conviction can we recover from the data alone?**

This notebook is designed to be useful for both practitioners and Kaggle readers:

- it surfaces non-obvious cross-asset patterns
- it avoids low-signal EDA clutter
- it builds a fast, reproducible baseline
- it adds one meaningful modeling improvement
- it finishes with interpretable takeaways instead of just scores

## Why This Dataset Matters

Most public market datasets cover either:

- only equities
- only price history
- or only a narrow slice of technical indicators

Here we have a much broader lens. Because the table includes **returns, volatility, drawdowns, momentum, valuation proxies, quality metrics, and analyst fields**, it supports:

- cross-asset regime analysis
- factor-style screening
- explainable ML baselines
- idea generation for trading, ranking, or research workflows

## What You Will Learn

By the end of this notebook, you will have:

1. A compact view of where different asset classes sit on the **return-risk spectrum**
2. A few **"aha" moments** hidden in the 52-week position and sentiment-like features
3. A **clean classification target** for analyst conviction
4. A **baseline model** and one stronger tree-based improvement
5. A feature-importance view of **what the model is actually using**

## Mini Roadmap

- Setup and data loading
- Quick overview and cleaning
- High-signal EDA
- Key insights summary
- Feature engineering
- Baseline model
- Improved model
- Validation and comparison
- Explainability
- Final takeaways, pitfalls, and next steps


In [ ]:
# ============================================
# Setup
# ============================================
import os
import glob
import random
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.frameon"] = True

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 120)


## Data Loading

Kaggle notebooks should run without manual edits, so the loader below searches under `/kaggle/input` and picks the market dataset automatically. A local fallback is included only for portability outside Kaggle.


In [ ]:
def find_dataset_path():
    kaggle_candidates = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
    preferred = [
        p for p in kaggle_candidates
        if "yahoo_finance_global_markets_2026" in os.path.basename(p).lower()
    ]
    if preferred:
        return preferred[0]
    if kaggle_candidates:
        return kaggle_candidates[0]

    local_candidates = sorted(glob.glob("*.csv"))
    preferred_local = [
        p for p in local_candidates
        if "yahoo_finance_global_markets_2026" in os.path.basename(p).lower()
    ]
    if preferred_local:
        return preferred_local[0]
    if local_candidates:
        return local_candidates[0]
    raise FileNotFoundError("No CSV dataset found under /kaggle/input or the working directory.")


DATA_PATH = find_dataset_path()
df = pd.read_csv(DATA_PATH)

print(f"Using dataset: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head(3))


## Quick Overview

Before modeling anything, it helps to understand two things:

- how broad the dataset really is
- where the missingness lives

With cross-asset data, missing values are not necessarily bad. They often reflect structural differences:

- crypto and forex do not have equity-style fundamentals
- ETFs do not carry the same sector fields as stocks
- analyst coverage is concentrated in equities


In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

print("Top columns by missingness")
display(summary.head(20))

overview = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "numeric_columns": df.select_dtypes(include=np.number).shape[1],
    "categorical_columns": df.select_dtypes(exclude=np.number).shape[1],
    "asset_classes": df["asset_class"].nunique(dropna=True),
    "quote_types": df["quoteType"].nunique(dropna=True),
}

pd.Series(overview, name="dataset_overview").to_frame()


## Data Cleaning

This is a relatively clean table already, so the goal is not aggressive cleaning. Instead, we apply a few practical safeguards:

- standardize column names and text labels
- parse dates defensively
- preserve NaNs for models that will handle them through imputation
- create a small number of engineered fields for analysis and modeling


In [ ]:
df = df.copy()
df.columns = [c.strip() for c in df.columns]

for col in ["price_date", "build_timestamp", "exDividendDate", "lastFiscalYearEnd", "mostRecentQuarter", "nextFiscalYearEnd"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Lightweight engineered fields used later in both EDA and modeling.
if {"price_vs_sma50_pct", "price_vs_sma200_pct"}.issubset(df.columns):
    df["trend_acceleration"] = df["price_vs_sma50_pct"] - df["price_vs_sma200_pct"]

if {"momentum_score", "volatility_1y_ann"}.issubset(df.columns):
    df["risk_adjusted_momentum"] = df["momentum_score"] / (df["volatility_1y_ann"].abs() + 1)

if {"pct_of_52w_high", "max_drawdown_1y_pct"}.issubset(df.columns):
    df["drawdown_resilience"] = df["pct_of_52w_high"] + df["max_drawdown_1y_pct"]

if {"quality_score", "value_score"}.issubset(df.columns):
    df["quality_value_blend"] = np.log1p(df["quality_score"].clip(lower=0)) - np.log1p(df["value_score"].clip(lower=0))

print("Cleaning complete.")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")


## EDA

The next few sections focus on **signal, not volume**. Instead of plotting everything, we ask a few targeted questions:

1. Which asset classes delivered the best return-to-risk tradeoff?
2. Does the 52-week position actually encode useful information?
3. Do sentiment-like composite signals line up with realized performance?
4. Among covered equities, what does a BUY-rated profile look like?


In [ ]:
asset_metrics = (
    df.groupby("asset_class", dropna=False)
      .agg(
          n=("ticker", "count"),
          avg_return_1y=("return_1y_pct", "mean"),
          median_return_1y=("return_1y_pct", "median"),
          avg_sharpe_1y=("sharpe_1y", "mean"),
          avg_volatility_1y=("volatility_1y_ann", "mean"),
          avg_composite_score=("composite_score", "mean"),
      )
      .sort_values("avg_sharpe_1y", ascending=False)
)

display(asset_metrics.style.background_gradient(cmap="YlGn", subset=["avg_return_1y", "avg_sharpe_1y", "avg_composite_score"])
                         .background_gradient(cmap="YlOrRd_r", subset=["avg_volatility_1y"]))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=asset_metrics.reset_index(),
    x="asset_class",
    y="avg_return_1y",
    palette="viridis",
    ax=axes[0]
)
axes[0].set_title("Average 1Y Return by Asset Class")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=asset_metrics.reset_index(),
    x="asset_class",
    y="avg_sharpe_1y",
    palette="mako",
    ax=axes[1]
)
axes[1].set_title("Average 1Y Sharpe by Asset Class")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### Insight 1

A simple return ranking would miss something important:

- **Indices and ETFs look stronger on a risk-adjusted basis than many stock buckets**
- **Crypto is the clear outlier on volatility**
- **International equities are surprisingly competitive on both return and Sharpe**

That is already a good reminder that broad, diversified exposures can look better than high-beta narratives when we care about *quality of returns*, not just raw upside.


In [ ]:
zone_stats = (
    df.groupby("week52_zone", dropna=False)
      .agg(
          n=("ticker", "count"),
          avg_return_1y=("return_1y_pct", "mean"),
          median_return_1y=("return_1y_pct", "median"),
          avg_sharpe_1y=("sharpe_1y", "mean"),
          avg_volatility_1y=("volatility_1y_ann", "mean"),
          avg_composite_score=("composite_score", "mean")
      )
      .sort_values("avg_return_1y", ascending=False)
)
display(zone_stats)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

order = zone_stats.index.tolist()
sns.barplot(data=df, x="week52_zone", y="return_1y_pct", order=order, palette="crest", estimator=np.mean, errorbar=None, ax=axes[0])
axes[0].set_title("Average 1Y Return by 52-Week Zone")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=df, x="week52_zone", y="sharpe_1y", order=order, palette="flare", estimator=np.mean, errorbar=None, ax=axes[1])
axes[1].set_title("Average 1Y Sharpe by 52-Week Zone")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


### Insight 2: One of the strongest signals in the table is hiding in plain sight

The 52-week position is not just cosmetic.

Assets in the **`NEAR_HIGH`** bucket materially outperform those in **`NEAR_LOW`** on both:

- average 1Y return
- average Sharpe ratio

This is a strong momentum-style pattern and one of the notebook's first **aha moments**:

> **being close to the 52-week high looks more like persistent strength than overextension in this dataset**


In [ ]:
sentiment_stats = (
    df.groupby("fear_greed_label", dropna=False)
      .agg(
          n=("ticker", "count"),
          avg_return_1y=("return_1y_pct", "mean"),
          avg_sharpe_1y=("sharpe_1y", "mean"),
          avg_volatility_1y=("volatility_1y_ann", "mean"),
          avg_composite_score=("composite_score", "mean")
      )
      .sort_values("avg_return_1y", ascending=False)
)
display(sentiment_stats)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=df,
    x="fear_greed_label",
    y="return_1y_pct",
    order=sentiment_stats.index.tolist(),
    palette="Spectral",
    estimator=np.mean,
    errorbar=None,
    ax=axes[0]
)
axes[0].set_title("Average 1Y Return by Fear/Greed Label")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.scatterplot(
    data=df,
    x="pct_of_52w_high",
    y="sharpe_1y",
    hue="fear_greed_label",
    size="volatility_1y_ann",
    palette="Spectral",
    alpha=0.75,
    ax=axes[1]
)
axes[1].set_title("Fear/Greed, 52W Position, and Risk-Adjusted Returns")
axes[1].set_xlabel("% of 52-Week High")
axes[1].set_ylabel("1Y Sharpe")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


### Insight 3: Sentiment is telling us more about *stability* than about hype

Another useful surprise:

- **GREED / EXTREME_GREED** names are not just higher return, they are also lower volatility on average
- **FEAR** names combine weak returns with very high realized volatility

That suggests the sentiment proxy is partly capturing **trend health** and **drawdown pressure**, not just speculative enthusiasm.


In [ ]:
corr_features = [
    "return_1y_pct",
    "sharpe_1y",
    "max_drawdown_1y_pct",
    "volatility_1y_ann",
    "pct_of_52w_high",
    "fear_greed_proxy",
    "momentum_score",
    "composite_score",
    "quality_score",
    "value_score",
    "rsi_14",
]
corr_df = df[corr_features].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="RdYlGn", center=0)
plt.title("Correlation Map of High-Signal Numeric Features")
plt.show()


### Insight 4: Composite score is mostly a momentum story

The correlation view makes an important modeling point:

- `composite_score` tracks strongly with `momentum_score`
- `pct_of_52w_high`, drawdown, and fear/greed are all tightly linked

In other words, several columns describe overlapping parts of the same market state. That is useful for ranking, but it also means we should be careful not to over-interpret every feature as fully independent.


## Building a Practical Classification Target

For modeling, I want something that is:

- realistic
- interpretable
- not trivially leaked

So instead of predicting returns that were already used to compute many technical indicators, I focus on a cleaner task:

> **Can we predict whether an analyst-covered equity is rated BUY/STRONG_BUY versus HOLD/UNDERPERFORM/SELL using market and fundamental features alone?**

Important guardrail:

- I explicitly **drop analyst-derived columns** like target prices and recommendation scores from the feature set

This makes the exercise much more honest.


In [ ]:
model_df = df.copy()

valid_recs = ["buy", "strong_buy", "hold", "underperform", "sell"]
model_df = model_df[
    (model_df["quoteType"] == "EQUITY") &
    (model_df["recommendationKey"].isin(valid_recs))
].copy()

model_df["target_buy"] = model_df["recommendationKey"].isin(["buy", "strong_buy"]).astype(int)

print(f"Modeling sample: {model_df.shape[0]} rows")
print("Class balance:")
display(model_df["target_buy"].value_counts(normalize=True).rename("share").mul(100).round(1).to_frame())

plt.figure(figsize=(8, 4))
sns.countplot(data=model_df, x="recommendationKey", order=model_df["recommendationKey"].value_counts().index, palette="Set2")
plt.title("Recommendation Distribution in the Modeling Sample")
plt.xlabel("")
plt.show()


In [ ]:
buy_profile = (
    model_df.groupby("target_buy")
            .agg(
                n=("ticker", "count"),
                avg_return_1y=("return_1y_pct", "mean"),
                avg_sharpe_1y=("sharpe_1y", "mean"),
                avg_pct_of_52w_high=("pct_of_52w_high", "mean"),
                avg_momentum_score=("momentum_score", "mean"),
                avg_quality_score=("quality_score", "mean"),
                avg_value_score=("value_score", "mean"),
                avg_debt_to_equity=("debtToEquity", "mean"),
            )
)
display(buy_profile)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=model_df, x="target_buy", y="momentum_score", palette="coolwarm", ax=axes[0])
axes[0].set_title("BUY vs HOLD+: Momentum Score")
axes[0].set_xlabel("target_buy")

sns.boxplot(data=model_df, x="target_buy", y="pct_of_52w_high", palette="coolwarm", ax=axes[1])
axes[1].set_title("BUY vs HOLD+: % of 52W High")
axes[1].set_xlabel("target_buy")

sns.boxplot(data=model_df, x="target_buy", y="debtToEquity", palette="coolwarm", ax=axes[2])
axes[2].set_title("BUY vs HOLD+: Debt to Equity")
axes[2].set_xlabel("target_buy")

plt.tight_layout()
plt.show()


## Key Insights Summary

Before moving to ML, here are the main findings so far:

- **International equities, ETFs, and indices** look stronger than expected on a risk-adjusted basis
- **Near-high assets** dominate near-low assets in both return and Sharpe, which is a strong momentum confirmation
- The **fear/greed proxy** appears to encode trend health and drawdown pressure, not just speculative mood
- In the labeled equity subset, **BUY-rated names tend to have stronger momentum and sit closer to their 52-week highs**

These patterns give us a reasonable basis for a compact predictive model.


## Feature Engineering

This notebook intentionally keeps feature engineering lightweight and explainable.

Added features:

- `trend_acceleration`: short-term trend minus long-term trend
- `risk_adjusted_momentum`: momentum scaled by volatility
- `drawdown_resilience`: 52-week position combined with drawdown
- `quality_value_blend`: a simple balance between quality and valuation proxies

Just as important is what we **exclude**:

- `recommendationMean`
- `recommendationKey`
- `analyst_consensus`
- target price columns
- `numberOfAnalystOpinions`
- `analyst_upside_pct`

Those fields would make the task artificially easy.


In [ ]:
leak_prone_columns = [
    "ticker", "shortName", "longName", "symbol",
    "recommendationKey", "recommendationMean", "analyst_consensus",
    "targetHighPrice", "targetLowPrice", "targetMeanPrice", "targetMedianPrice",
    "numberOfAnalystOpinions", "analyst_upside_pct",
    "build_timestamp", "price_date",
    "target_buy",
]

candidate_features = [c for c in model_df.columns if c not in leak_prone_columns]
X = model_df[candidate_features].copy()
y = model_df["target_buy"].copy()

# Remove datetime columns for sklearn compatibility.
datetime_cols = X.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()
X = X.drop(columns=datetime_cols, errors="ignore")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print(f"Feature matrix shape: {X.shape}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


## Validation Strategy

With only a few hundred labeled rows, we should avoid noisy one-shot validation. I use:

- a **stratified train/test split** for a clean final holdout
- **repeated stratified cross-validation** on the training set for model comparison

Metrics:

- **ROC AUC** for ranking quality
- **F1** for class balance awareness
- **Balanced accuracy** because the positive class dominates


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


## Baseline Model

A good baseline should be:

- strong enough to be credible
- simple enough to be interpretable
- fast enough to rerun easily

Logistic regression is a good fit here. It handles many features well once we impute, scale, and one-hot encode.


In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED))
])


def evaluate_model(name, estimator, X_train, y_train, X_test, y_test, cv):
    scoring = {
        "roc_auc": "roc_auc",
        "f1": "f1",
        "balanced_accuracy": "balanced_accuracy",
        "accuracy": "accuracy",
    }
    cv_result = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=None,
        return_train_score=False,
    )

    fitted = clone(estimator).fit(X_train, y_train)
    test_pred = fitted.predict(X_test)
    test_proba = fitted.predict_proba(X_test)[:, 1]

    result = {
        "Model": name,
        "CV ROC AUC": np.mean(cv_result["test_roc_auc"]),
        "CV F1": np.mean(cv_result["test_f1"]),
        "CV Balanced Acc": np.mean(cv_result["test_balanced_accuracy"]),
        "Holdout ROC AUC": roc_auc_score(y_test, test_proba),
        "Holdout F1": f1_score(y_test, test_pred),
        "Holdout Balanced Acc": balanced_accuracy_score(y_test, test_pred),
        "Holdout Accuracy": accuracy_score(y_test, test_pred),
    }
    return fitted, result


baseline_fitted, baseline_result = evaluate_model(
    "Logistic Regression",
    baseline_model,
    X_train, y_train, X_test, y_test, cv
)

pd.DataFrame([baseline_result]).round(4)


## Improved Model

The baseline is linear. That is useful, but market signals often interact in non-linear ways:

- strong momentum matters more when volatility is controlled
- quality and leverage can combine differently across sectors
- the same RSI level may mean something different depending on trend context

For that reason, the improvement step uses a **Random Forest** on the same cleaned feature space. It is still fast, robust to mixed inputs after preprocessing, and much better at capturing interactions.


In [ ]:
improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    ))
])

improved_fitted, improved_result = evaluate_model(
    "Random Forest",
    improved_model,
    X_train, y_train, X_test, y_test, cv
)

results_df = pd.DataFrame([baseline_result, improved_result]).sort_values("Holdout ROC AUC", ascending=False)
display(results_df.round(4))


## Model Comparison

The comparison table below is intentionally simple:

| Model | Score | Notes |
|---|---:|---|
| Logistic Regression | ROC AUC / F1 | Strong baseline, linear, interpretable |
| Random Forest | ROC AUC / F1 | Captures non-linear interactions and feature thresholds |

In a small tabular setting like this, that is exactly the kind of progression I want:

- **start clean**
- **add one meaningful improvement**
- **avoid overcomplicating the notebook**


In [ ]:
comparison = results_df.copy()
comparison["Notes"] = comparison["Model"].map({
    "Logistic Regression": "Stable baseline after scaling and one-hot encoding",
    "Random Forest": "Best overall balance by modeling non-linear interactions",
})
display(comparison.round(4))


## Holdout Validation

Raw scores matter, but so does failure mode. The next diagnostics show:

- confusion matrix
- precision/recall balance
- whether the stronger model is simply overpredicting BUY


In [ ]:
best_model = improved_fitted if improved_result["Holdout ROC AUC"] >= baseline_result["Holdout ROC AUC"] else baseline_fitted
best_name = improved_result["Model"] if improved_result["Holdout ROC AUC"] >= baseline_result["Holdout ROC AUC"] else baseline_result["Model"]

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title(f"Confusion Matrix: {best_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.show()

display(pd.DataFrame(report).T.round(3))


## Explainability

A model score without interpretation is rarely enough for a public notebook.

I am using **permutation importance** instead of a more complex explainability stack because it is:

- model-agnostic
- easy to understand
- fast enough for Kaggle runtime constraints


In [ ]:
transformed_feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
transformed_X_test = best_model.named_steps["preprocessor"].transform(X_test)

perm = permutation_importance(
    best_model.named_steps["model"],
    transformed_X_test,
    y_test,
    n_repeats=20,
    random_state=SEED,
    scoring="roc_auc"
)

importance_df = (
    pd.DataFrame({
        "feature": transformed_feature_names,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

top_features = importance_df.head(15)
display(top_features)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_features.sort_values("importance_mean"), x="importance_mean", y="feature", palette="viridis")
plt.title(f"Permutation Importance: {best_name}")
plt.xlabel("Mean ROC AUC Drop")
plt.ylabel("")
plt.show()


### Interpreting the Model

In most runs, the most useful features should come from a compact mix of:

- trend strength
- 52-week position
- risk-adjusted momentum
- quality / leverage
- a few broad categorical context fields such as sector or market-cap tier

That is a useful practical result. Analyst conviction is not random here. It is strongly aligned with a combination of:

- **where the asset sits in its trend**
- **how healthy the return profile looks**
- **whether the underlying business quality supports that price behavior**


## Error Analysis

This is one of the easiest ways to make a notebook more useful.

Instead of stopping at feature importance, let's inspect the holdout cases where the model was most confidently wrong. Those rows often reveal:

- analyst discretion that the features cannot capture
- sector-specific narratives
- valuation or event risk hidden behind strong technicals


In [ ]:
error_analysis = X_test.copy()
error_analysis["actual"] = y_test.values
error_analysis["predicted"] = y_pred
error_analysis["predicted_proba_buy"] = y_proba
error_analysis["confidence"] = np.where(y_pred == 1, y_proba, 1 - y_proba)
error_analysis["correct"] = (error_analysis["actual"] == error_analysis["predicted"]).astype(int)

hard_cases = error_analysis[error_analysis["correct"] == 0].sort_values("confidence", ascending=False).head(10)
display(hard_cases[[
    c for c in [
        "asset_class", "sector", "market_cap_tier", "momentum_score", "pct_of_52w_high",
        "quality_score", "value_score", "debtToEquity", "actual", "predicted", "predicted_proba_buy"
    ] if c in hard_cases.columns
]])


## Pitfalls Section

A good finance notebook should also be honest about limits.

What worked:

- concise cross-asset segmentation
- leakage-aware target design
- a simple baseline + one stronger non-linear model
- explainability that stays readable

What did not fully work:

- the labeled sample is relatively small
- analyst coverage is concentrated in equities, so this is not a universal cross-asset classifier
- some engineered scores likely overlap heavily, which can blur causal interpretation

Important caveat:

- this notebook is best read as **market intelligence + ranking insight**, not as a production trading system


## Final Insights

Three takeaways stand out:

1. **Position in the 52-week range is one of the strongest high-level signals in the dataset.**
2. **Cross-asset breadth matters.** Some of the best risk-adjusted profiles come from indices, ETFs, and international names rather than the noisiest high-volatility assets.
3. **Analyst BUY ratings are partially predictable from observable market structure and business quality**, even after removing analyst-derived leakage columns.

That makes this dataset especially useful for:

- screening workflows
- market regime dashboards
- factor-style experiments
- ranking and recommendation research


## Next Steps

If I were extending this notebook, I would try:

- a **regression notebook** for `return_1y_pct` with stronger leakage controls
- **segment-specific models** for US mega caps vs international names
- **clustering** on risk, trend, and quality to build market archetypes
- **time-aware snapshots** if future versions of the dataset expose multiple dates

A promising Kaggle follow-up would be to turn this into a **cross-asset ranking notebook** that surfaces the most attractive names under different risk appetites.


## Conclusion

This dataset is much more than a feature dump. Used carefully, it can tell a compact but powerful story about:

- where market strength is concentrated
- how technical and risk signals cluster together
- and which observable traits most often align with analyst conviction

The main practical lesson is simple:

> **Strong market profiles tend to live near highs, carry healthier risk-adjusted momentum, and show up consistently across multiple related signals.**

If you found this useful, feel free to upvote 👍
